# 09 打造 Bantay Santa Luz 儀表板 — 筆記本

現在是 2026 年 9 月 30 日。市長要給地方衛生委員會看**一個畫面**：有多少病例、在哪裡、往上還是往下，以及我們要做什麼。在這本 筆記本 裡，Maya 在 Tala 的協助下，一步一步做出這個畫面的簡化版：

1. 計算關鍵數字（KPI）；
2. 做三張 Plotly 圖：流行曲線、地圖、登革熱對照它的閾值；
3. 用一個小範本把所有東西放進**一個獨立的 HTML 檔**；
4. 在任何東西離開辦公室之前，先跑 Doc Mon 的**公開前檢查**。

這裡的一切都由合成的 Santa Luz 資料算出。Santa Luz、它的居民與它的數字都是虛構的。我們寫出的檔案會放在你電腦的暫存資料夾，不會放進書裡。

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## 步驟 1 · 載入資料

我們需要四個檔案：乾淨、已去識別化的造冊、各 barangay（描籠涯，菲律賓最基層的行政區，類似台灣的村里）的人口、每週登革熱病例數，以及 barangay 地圖（GeoJSON）。它們都沒有姓名或地址——第 03 章已經把那些移除了。

In [ ]:
import importlib.util
import json
import re
import tempfile
from pathlib import Path
from string import Template

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

DATA = Path("data/synthetic/ph")
ll = pd.read_csv(DATA / "santa_luz_linelist.csv", parse_dates=["date_onset"])
barangays = pd.read_csv(DATA / "santa_luz_barangays.csv")
weekly = pd.read_csv(DATA / "santa_luz_dengue_weekly.csv")
geojson = json.loads((DATA / "santa_luz_barangays.geojson").read_text(encoding="utf-8"))
AS_OF = "2026 年 9 月 30 日"
AGE_GROUPS = ["0-4", "5-14", "15-24", "25-44", "45-64", "65+"]

print(len(ll), "例 ·", len(barangays), "個 barangay ·", len(weekly), "週的登革熱病例數")

## 步驟 2 · 有多少？關鍵數字

Maya 問 Tala：

> *我有一個 pandas DataFrame `ll`，每一列是一位病例，欄位有 `case_id`、`disease`（`leptospirosis` 或 `dengue`）、`outcome`（`alive` 或 `died`）和 `hospitalised`（0/1）。虛構的列：`X-1, dengue, alive, 0` · `X-2, leptospirosis, died, 1` · `X-3, dengue, alive, 1`。請寫一個函式 `compute_kpis(ll)`，針對每一種疾病以及全市，回傳病例數、死亡數、住院數，以及致死率（%，取到小數一位）。*

答案是這樣：

In [ ]:
def compute_kpis(ll: pd.DataFrame) -> pd.DataFrame:
    """各疾病的病例數、死亡數、住院數與致死率（%），再加上全市總計列。"""
    kpis = ll.groupby("disease").agg(
        cases=("case_id", "size"),
        deaths=("outcome", lambda s: (s == "died").sum()),
        hospitalised=("hospitalised", "sum"),
    )
    kpis.loc["total"] = kpis.sum()
    kpis["cfr_pct"] = (100 * kpis["deaths"] / kpis["cases"]).round(1)
    return kpis


kpis = compute_kpis(ll)
kpis

**檢查一下。** 兩項快速檢查：在三列我們能手算的資料上執行這個函式，再用第二種不同的方式數死亡人數。致死率的分母必須是*病例數*，而且總計列必須加得起來。

In [ ]:
tiny = pd.DataFrame({"case_id": ["X-1", "X-2", "X-3"],
                     "disease": ["dengue", "leptospirosis", "dengue"],
                     "outcome": ["alive", "died", "alive"],
                     "hospitalised": [0, 1, 1]})
tiny_kpis = compute_kpis(tiny)
assert tiny_kpis.loc["dengue", "cases"] == 2 and tiny_kpis.loc["leptospirosis", "cfr_pct"] == 100
assert kpis.loc["total", "deaths"] == (ll["outcome"] == "died").sum()
assert kpis.loc["total", "cases"] == len(ll)
print("檢查通過。")

## 步驟 3 · 我們到處都會用的隱私規則

儀表板會送到 CESU 以外的人手上。在小群體（一個 barangay、一週、一個年齡組）裡，1 到 4 的數字可能指向某位鄰居。所以，在畫出或印出任何東西之前，1 到 4 的數字都會變成 **「<5」**，並且不公布它們的侵襲率。0 維持 0：它沒有描述任何人。整個期間的全市總數（KPI）會完整公布，就像例行公報那樣。

In [ ]:
def suppress(counts: pd.Series | pd.DataFrame, threshold: int = 5) -> pd.Series | pd.DataFrame:
    """把 1 到 threshold - 1 的數字換成 '<5'（0 維持 0）。"""
    small = (counts > 0) & (counts < threshold)
    return counts.astype(object).mask(small, f"<{threshold}")


suppress(pd.Series([0, 1, 4, 5, 12]))

## 步驟 4 · 什麼時候？流行曲線

依發病**週別**（ISO 週）計算的病例數，兩種疾病疊在一起。週別夠粗，不會洩漏任何人確切的發病日。某一週某種疾病只有 1–4 例時，那根長條會留白，並在滑鼠懸停的提示文字裡標示為「<5」。我們標出颱風 Amihan（7 月 18 日，在第 29 週）。

In [ ]:
ll["week"] = ll["date_onset"].dt.isocalendar().week
DISEASE_ZH = {"leptospirosis": "鉤端螺旋體病", "dengue": "登革熱"}   # 顯示用的疾病名稱
curve = pd.crosstab(ll["week"], ll["disease"]).reindex(range(28, 40), fill_value=0)
public_curve = curve.mask((curve > 0) & (curve < 5))      # 1-4：不畫
labels = [f"W{w}" for w in curve.index]

fig_curve = go.Figure()
for disease, colour in [("leptospirosis", "#0038A8"), ("dengue", "#E8590C")]:
    fig_curve.add_bar(x=labels, y=public_curve[disease], name=DISEASE_ZH[disease],
                      marker_color=colour, customdata=suppress(curve[disease]),
                      hovertemplate="%{x}：%{customdata} 例<extra></extra>")
fig_curve.add_annotation(x="W29", y=curve.sum(axis=1).max(), text="颱風 Amihan，7/18",
                         showarrow=True, arrowhead=2, ay=-30)
fig_curve.update_layout(barmode="stack", bargap=0.1, height=380,
                        title="依發病週別的病例數，2026 年",
                        xaxis_title="發病的 ISO 週", yaxis_title="病例數")
fig_curve

In [ ]:
# 檢查：曲線保留了每一個有日期的病例（有 2 例沒有發病日）
assert curve.to_numpy().sum() == ll["date_onset"].notna().sum() == len(ll) - 2
print(curve.loc[30:34, "leptospirosis"].tolist(), "<- 鉤端螺旋體病 W30-W34；README：35, 108, 68, 15, 2")

## 步驟 5 · 在哪裡？登革熱侵襲率地圖

病例數畫在地圖上會誤導人，因為大的 barangay 病例比較多。我們畫**侵襲率**：病例數 ÷ 人口 × 10,000。登革熱病例是 1–4 例的 barangay 會顯示「<5」，而且**不列侵襲率**（侵襲率加上人口就能把病例數算回來）。

In [ ]:
counts = (ll.groupby(["barangay_id", "disease"]).size().unstack(fill_value=0)
          .reindex(barangays["barangay_id"], fill_value=0))
table = barangays.set_index("barangay_id")[["barangay", "band", "population"]].join(counts)
table["dengue_rate"] = (10_000 * table["dengue"] / table["population"]).round(1)
table.loc[table["dengue"].between(1, 4), "dengue_rate"] = None   # 不公布小格的侵襲率
table["dengue_shown"] = suppress(table["dengue"])

fig_map = px.choropleth(
    table.reset_index(), geojson=geojson, locations="barangay_id",
    featureidkey="properties.barangay_id", color="dengue_rate",
    color_continuous_scale="Oranges", hover_name="barangay",
    hover_data={"barangay_id": False, "dengue_shown": True, "dengue_rate": True},
    labels={"dengue_rate": "每 10,000 人", "dengue_shown": "病例數"},
    title="每 10,000 人登革熱侵襲率（灰色：少於 5 例，不列侵襲率）")
fig_map.update_geos(fitbounds="locations", visible=False)
fig_map.update_layout(height=420, margin={"l": 0, "r": 0, "t": 50, "b": 0})
fig_map

In [ ]:
# 對照 README 檢查最高的侵襲率：Talaba 43.4、Buhangin 35.5
print(table.sort_values("dengue_rate", ascending=False)[["barangay", "dengue_shown", "dengue_rate"]].head(3))
print("登革熱侵襲率被遮蔽的 barangay 數：", table["dengue_rate"].isna().sum())

## 步驟 6 · 往上還是往下？登革熱對照流行閾值

每一個 ISO 週的閾值，是平常年份 2022–2025 年中，該週及其前後兩週的**平均值 + 2 個標準差**（第 06 章）。我們從第 22 週開始畫 2026 年。

In [ ]:
base = weekly[weekly["iso_year"].between(2022, 2025)]


def threshold_for(week: int) -> float:
    """基準年中，第 week-1、week、week+1 週的平均值 + 2 個標準差。"""
    window = [(week - 2) % 52 + 1, week, week % 52 + 1]
    values = base.loc[base["iso_week"].isin(window), "cases"]
    return values.mean() + 2 * values.std()


y2026 = weekly[(weekly["iso_year"] == 2026) & (weekly["iso_week"] >= 22)].copy()
y2026["threshold"] = y2026["iso_week"].map(threshold_for)
above = y2026[y2026["cases"] > y2026["threshold"]]
print(len(above), "週高於閾值，第一週：", above["morbidity_week"].iloc[0])

shown = y2026["cases"].mask(y2026["cases"].between(1, 4))
fig_thr = go.Figure()
fig_thr.add_bar(x=y2026["morbidity_week"], y=shown, name="登革熱病例數", marker_color="#E8590C",
                customdata=suppress(y2026["cases"]),
                hovertemplate="%{x}：%{customdata} 例<extra></extra>")
fig_thr.add_scatter(x=y2026["morbidity_week"], y=y2026["threshold"], name="流行閾值",
                    mode="lines", line={"color": "#CE1126", "dash": "dash", "shape": "hvh"})
fig_thr.update_layout(height=360, title="每週登革熱病例數對照流行閾值，2026 年",
                      yaxis_title="病例數")
fig_thr

In [ ]:
# 對照 README 檢查：2026-W32 首度高於閾值，33 例對閾值 17.5
w32 = y2026.set_index("morbidity_week").loc["2026-W32"]
assert len(above) == 8 and w32["cases"] == 33 and round(w32["threshold"], 1) == 17.5
print("閾值檢查通過。注意：W39 不完整（通報延遲）。")

## 步驟 7 · 一個獨立的 HTML 檔

Maya 請 Tala 做「一個小小的 HTML 範本，有標題、*資料截至*日期、KPI 方塊、三張圖，以及資料說明區，寫成一個離線也能用的檔案」。Python 的 `string.Template` 會填入 `$placeholders`。`fig.to_html(full_html=False, ...)` 把每張圖變成一段 HTML 片段；第一張帶有 Plotly 函式庫本身（`include_plotlyjs=True`），所以這個檔案不需要網路。

In [ ]:
PAGE = Template("""<!DOCTYPE html>
<html lang="zh-Hant"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>$title</title>
<style>
  body { font-family: system-ui, 'Noto Sans TC', 'PingFang TC', 'Microsoft JhengHei', sans-serif; max-width: 1100px; margin: 0 auto; padding: 12px;
         background: #fffbf2; color: #1c1b29; }
  header { background: #0038A8; color: #FCD116; padding: 10px 16px; border-radius: 12px; }
  header p { color: #fff; margin: 0; }
  .kpis { display: flex; flex-wrap: wrap; gap: 10px; margin: 12px 0; }
  .kpi { flex: 1 1 140px; background: #fff; border: 2px solid #eadcc0; border-radius: 12px;
         padding: 8px 12px; }
  .kpi b { display: block; font-size: 28px; color: #0038A8; }
  .notes { font-size: 14px; color: #4d4a5e; }
</style></head><body>
<header><h1>$title</h1><p>資料截至 $as_of · 合成教學資料</p></header>
<section class="kpis">$kpis</section>
$figures
<section class="notes"><h2>資料說明</h2><ul>$notes</ul></section>
</body></html>""")

tiles = [("病例數", kpis.loc["total", "cases"]),
         ("死亡數", kpis.loc["total", "deaths"]),
         ("致死率 · 鉤端螺旋體病", f'{kpis.loc["leptospirosis", "cfr_pct"]}%'),
         ("致死率 · 登革熱", f'{kpis.loc["dengue", "cfr_pct"]}%'),
         ("高於登革熱閾值的週數", len(above))]
notes = ["Santa Luz 市與它所有的資料都是合成資料，為教學而製作。",
         "只呈現彙總數字：沒有姓名、地址、病例編號，也沒有個別病例的確切日期。",
         "任何 barangay 或任何一週中 1-4 的數字顯示為 <5，並且不公布它們的侵襲率。",
         "病例依發病週別計算；最後一週不完整（通報延遲）。"]

figures = [fig.to_html(full_html=False, include_plotlyjs=(i == 0), config={"responsive": True})
           for i, fig in enumerate([fig_curve, fig_map, fig_thr])]
page_html = PAGE.substitute(
    title="Bantay Santa Luz",
    as_of=AS_OF,
    kpis="".join(f'<div class="kpi">{label}<b>{value}</b></div>' for label, value in tiles),
    figures="\n".join(figures),
    notes="".join(f"<li>{note}</li>" for note in notes),
)
out = Path(tempfile.gettempdir()) / "bantay_santa_luz_simple.html"
out.write_text(page_html, encoding="utf-8")
print(f"已寫入 {out}（{out.stat().st_size / 1e6:.1f} MB）。請用瀏覽器開啟，再用手機開一次。")

## 步驟 8 · Doc Mon 的公開前檢查

在檔案離開 CESU 之前，Doc Mon 會逐項檢查一份清單。其中有些項目可以用程式碼檢查；把它們變成一個函式，這樣你就能在每個版本上執行：

- 只放彙總資料——沒有病例編號，沒有造冊欄位；
- 小格已遮蔽——沒有任何表格儲存格是 1–4；
- 有*資料截至*日期，以及說明這些數字是什麼的資料說明。

（有些項目仍須由人來做：文字裡沒有姓名或確切地址、誠實的座標軸、在手機上測試過。）

In [ ]:
def publish_check(page: str) -> list[str]:
    """回傳 Doc Mon 會擋下的問題（空清單代表可以公開）。"""
    problems = []
    if re.search(r"SLC-\d{4}", page):
        problems.append("發現病例編號：不得公開逐筆資料")
    for column in ["case_id", "date_onset", "date_report"]:
        if column in page:
            problems.append(f"發現造冊欄位 {column!r}")
    small = re.findall(r"<td>\s*[1-4]\s*</td>", page)
    if small:
        problems.append(f"{len(small)} 個表格儲存格是 1-4 例：請遮蔽")
    for must in ["資料截至", "資料說明", "合成資料"]:
        if must not in page:
            problems.append(f"缺少：{must!r}")
    return problems


print(publish_check(page_html) or "通過：可以交給 Doc Mon 做最後的人工檢查")

### Tala 搞錯了

市長的幕僚說想要「更多細節」，所以 Maya 請 Tala *加上依 barangay 與年齡組的病例表，再加一個下載資料的按鈕*。Tala 的程式碼看起來很整齊：

```python
detail = pd.crosstab(ll["barangay"], ll["age_group"])          # 原始計數，連 1 和 2 都有
download = f"<script>const DATA = {ll.to_json(orient='records')};</script>"  # 整份造冊！
bad_page = page_html.replace("</body>", detail.to_html() + download + "</body>")
```

我們來執行它——**只在記憶體裡**，絕不存檔——再交給公開前檢查。

In [ ]:
detail = pd.crosstab(ll["barangay"], ll["age_group"])
download = f"<script>const DATA = {ll.to_json(orient='records', date_format='iso')};</script>"
bad_page = page_html.replace("</body>", detail.to_html() + download + "</body>")
for problem in publish_check(bad_page):
    print("已擋下：", problem)

有兩個嚴重的問題。好幾個表格儲存格寫著 **1** 或 **2**：在一個只有幾千人的 barangay 裡，「一位 65 歲以上的病例」是鄰居叫得出名字的人，而不是一個統計數字。而那個「下載」把**造冊的每一列**——病例編號、年齡、確切的發病日——都放進了頁面裡，任何人用*檢視原始碼*就讀得到。解法：只公開彙總資料，而且要遮蔽小格。

In [ ]:
safe_detail = suppress(pd.crosstab(ll["band"], ll["age_group"])[AGE_GROUPS])  # 先彙總得更粗，再遮蔽小格
fixed_page = page_html.replace("</body>", safe_detail.to_html() + "</body>")
print(publish_check(fixed_page) or "通過")
safe_detail

## 步驟 9 · 完整的網站儀表板是同樣的想法，做得更深入

本書網站上的 [Bantay Santa Luz 儀表板](https://ancientsky.github.io/python4epi/ph/zh/dashboard/)，是由 `scripts/build_ph_dashboard.py` 用同樣的步驟建立的：`load_data()`、`compute_kpis()`、`suppress_small_cells()`、`render_html()`。它用一小段手寫的腳本而不是 Plotly 來畫圖表，所以檔案不到 0.2 MB。它的 `suppress_small_cells()` 還多了一條規則：當公布了某一欄的總數，而那一欄只遮蔽了一個小格時，會再多遮蔽第二格（`x`），這樣就沒有人能用減法把那個小數字算回來。

這個腳本是儲存庫的一部分（Colab 的那個儲存格已經把它複製下來了），所以我們可以匯入它：

In [ ]:
spec = importlib.util.spec_from_file_location("build_ph_dashboard", "scripts/build_ph_dashboard.py")
dashboard = importlib.util.module_from_spec(spec)
spec.loader.exec_module(dashboard)

full = dashboard.compute_kpis(dashboard.load_data())
assert full["cases"]["total"] == kpis.loc["total", "cases"]
assert full["cfr_pct"]["leptospirosis"] == kpis.loc["leptospirosis", "cfr_pct"]
print("和我們算出的 KPI 相同：", full["cases"], full["cfr_pct"])

dengue = ll[ll["disease"] == "dengue"]
dashboard.suppress_small_cells(pd.crosstab(dengue["age_group"], dengue["sex"]).reindex(AGE_GROUPS))

In [ ]:
# 把完整的儀表板建到你的暫存資料夾（約需一秒）
page = dashboard.build_dashboard("zh", Path(tempfile.gettempdir()) / "bantay_full")
print(page, f"{page.stat().st_size / 1e3:.0f} kB")
print(publish_check(page.read_text(encoding="utf-8")) or "通過")

## 換你試試

1. **加一個方塊。** 把住院病例數放進 `tiles`，重建 `page_html`，再跑一次公開前檢查。*提示：* `kpis.loc["total", "hospitalised"]`。
2. **畫另一種疾病的地圖。** 讓 `fig_map` 顯示鉤端螺旋體病的侵襲率。*提示：* 複製步驟 5 裡三行 `dengue_…`，把 `dengue` 改成 `leptospirosis`。哪個區帶亮起來？
3. **更嚴格的規則。** 有些單位會遮蔽小於 10 的數字。用 `threshold=10` 的話，有多少個 barangay 會失去它們的登革熱侵襲率？*提示：* `(suppress(table["dengue"], 10) == "<10").sum()`。

In [ ]:
# 換你試試 1：加一個住院病例數的方塊

In [ ]:
# 換你試試 2：畫鉤端螺旋體病的地圖

In [ ]:
# 換你試試 3：更嚴格的遮蔽規則

## 送出去之前

在**手機**上開啟這個 HTML 檔（寄給你自己，不要丟到群組聊天室），把每個數字大聲唸過一遍，然後請 Doc Mon——一個真人——給最後的同意。程式碼抓得到機械性的錯誤；其他的，要靠人來抓。